# CodeBERT AI detector — thử nghiệm nguồn tạo code Python

Đây KHÔNG phải notebook so sánh hai bài. Nhãn 0=human, 1=ai_generated.
Chọn **Settings → Accelerator → GPU**, bật **Internet**, rồi **Run All**.
Không cần API key, không tạo mã AI qua dịch vụ trả phí, không upload bài sinh viên.

Nguồn: [AIGCodeSet](https://huggingface.co/datasets/basakdemirok/AIGCodeSet),
revision b4d4e69fdbcba2608c99808c6afbf53a3bfd108c, giấy phép dataset theo publisher: CDLA-Permissive-2.0.
Trích dẫn Demirok & Kutlu, SIU 2025, arXiv:2412.16594; giữ DATASET_CARD.md khi chia sẻ kết quả.

Chỉ dùng AI `Generate`; bỏ AI sửa bài người. Chia theo nhóm đề, loại trùng token,
giữ LLAMA ngoài train/validation. Hai tập test dùng CHUNG mẫu người — không cộng gộp.
Một lần chạy, một seed chưa đủ để tuyên bố độ tin cậy. Không dùng kết quả để tự trừ điểm.


In [ ]:
%pip install -q "torch>=2.6,<3" "transformers==4.57.1" "scikit-learn>=1.5,<2" "numpy>=1.26,<3" "safetensors>=0.4.3"


In [ ]:
import json
import sys
import time
from pathlib import Path

ROOT = Path("/kaggle/working/ai_detector_experiment")
ROOT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT))


In [ ]:
(ROOT / 'prepare_data.py').write_text('"""Prepare public AIGCodeSet without executing any collected code (stdlib only)."""\n\nimport argparse\nimport csv\nimport hashlib\nimport io\nimport json\nimport random\nimport re\nimport tokenize\nimport urllib.request\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\n\nREPO = "basakdemirok/AIGCodeSet"\nREVISION = "b4d4e69fdbcba2608c99808c6afbf53a3bfd108c"\nFILES = ("human_selected_dataset.csv", "created_dataset_with_llms.csv")\nGENERATORS = {"CODESTRAL", "GEMINI", "LLAMA"}\nEXPECTED_SHA256 = {\n    "human_selected_dataset.csv": "f02f1099de41434e67f26dd0470e8cce32ac1a80966cea51c8f9a25b95991de8",\n    "created_dataset_with_llms.csv": "be858efdb3233dff32b4cf239171dd686f9fb968d26cb0ab44c668fa40b2226e",\n    "README.md": "366485d67d815f9a8ca8e6feaecca8fa615144615d595af38c6a8686c496470d",\n}\n\n\ndef digest(value):\n    return hashlib.sha256(value.encode("utf-8")).hexdigest()\n\n\ndef clean_code(code):\n    """Strip a whole-output Markdown fence symmetrically, not comments/docstrings."""\n    code = code.replace("\\r\\n", "\\n").replace("\\r", "\\n").strip()\n    match = re.fullmatch(\n        r"```(?:python|py)?\\s*\\n(.*?)\\n```", code, re.DOTALL | re.IGNORECASE\n    )\n    return (match.group(1).strip() if match else code) + "\\n"\n\n\ndef normalized_hash(code):\n    try:\n        tokens = [\n            (t.type, t.string)\n            for t in tokenize.generate_tokens(io.StringIO(code).readline)\n            if t.type\n            not in {\n                tokenize.COMMENT,\n                tokenize.NL,\n                tokenize.NEWLINE,\n                tokenize.INDENT,\n                tokenize.DEDENT,\n                tokenize.ENDMARKER,\n            }\n        ]\n        return digest(json.dumps(tokens, ensure_ascii=False))\n    except (tokenize.TokenError, IndentationError, SyntaxError):\n        # Retain malformed human code; don\'t make syntax validity a label filter.\n        return digest(code)\n\n\ndef read_sources(root):\n    records, excluded = [], Counter()\n    for filename in FILES:\n        with (root / filename).open(encoding="utf-8-sig", newline="") as handle:\n            reader = csv.DictReader(handle)\n            required = {\n                "problem_id",\n                "submission_id",\n                "LLM",\n                "status_in_folder",\n                "code",\n                "label",\n            }\n            if not required.issubset(reader.fieldnames or []):\n                raise ValueError(f"Unexpected schema in {filename}")\n            for index, row in enumerate(reader):\n                label = int(row["label"])\n                generator = row["LLM"].strip().upper()\n                if label not in (0, 1) or (label == 0) != (generator == "HUMAN"):\n                    raise ValueError("Label/source mismatch; do not infer or relabel")\n                if label == 1 and generator not in GENERATORS:\n                    raise ValueError(f"Unknown AI generator: {generator}")\n                if label == 1 and row["status_in_folder"].strip().lower() != "generate":\n                    excluded["ai_repair_or_hybrid"] += 1\n                    continue\n                if row.get("language", "Python").lower() != "python":\n                    excluded["non_python"] += 1\n                    continue\n                if not row["code"].strip() or not row["problem_id"].strip():\n                    excluded["empty_code_or_problem"] += 1\n                    continue\n                code = clean_code(row["code"])\n                records.append(\n                    {\n                        "record_id": f"{filename}:{index}",\n                        "code": code,\n                        "label": label,\n                        "problem_id": row["problem_id"].strip(),\n                        "generator": generator,\n                        "status": row["status_in_folder"],\n                        "sha256": digest(code),\n                        "normalized_hash": normalized_hash(code),\n                    }\n                )\n    return records, dict(excluded)\n\n\ndef make_splits(records, seed=42, heldout="LLAMA"):\n    """Group connected problems via duplicate code, then split before withholding LLM."""\n    if heldout not in GENERATORS:\n        raise ValueError("heldout must be a known generator")\n    parent = {r["problem_id"]: r["problem_id"] for r in records}\n\n    def find(x):\n        while parent[x] != x:\n            parent[x] = parent[parent[x]]\n            x = parent[x]\n        return x\n\n    by_hash = defaultdict(list)\n    for row in records:\n        by_hash[row["normalized_hash"]].append(row)\n    unique, conflicts = [], 0\n    for group in by_hash.values():\n        if len({r["label"] for r in group}) > 1:\n            conflicts += len(group)\n            continue  # Identical code with conflicting provenance is not identifiable.\n        for row in group[1:]:\n            parent[find(row["problem_id"])] = find(group[0]["problem_id"])\n        unique.append(min(group, key=lambda r: r["record_id"]))\n    groups = sorted({find(r["problem_id"]) for r in unique})\n    if len(groups) < 20:\n        raise ValueError("Need at least 20 independent problem groups")\n    random.Random(seed).shuffle(groups)\n    first, second = int(len(groups) * 0.65), int(len(groups) * 0.80)\n    assignment = {\n        g: ("train" if i < first else "validation" if i < second else "test")\n        for i, g in enumerate(groups)\n    }\n    splits = {name: [] for name in ("train", "validation", "test_seen", "test_unseen")}\n    excluded_heldout = 0\n    for row in unique:\n        row = dict(row, group_id=find(row["problem_id"]))\n        partition = assignment[row["group_id"]]\n        if partition != "test":\n            if row["generator"] == heldout:\n                excluded_heldout += 1\n            else:\n                splits[partition].append(row)\n        else:\n            if row["generator"] != heldout:\n                splits["test_seen"].append(row)\n            if row["label"] == 0 or row["generator"] == heldout:\n                splits["test_unseen"].append(row)\n    for name, rows in splits.items():\n        if {r["label"] for r in rows} != {0, 1}:\n            raise ValueError(\n                f"{name} lacks both labels; inspect data rather than silently changing split"\n            )\n    for left, right in (\n        ("train", "validation"),\n        ("train", "test_seen"),\n        ("train", "test_unseen"),\n        ("validation", "test_seen"),\n        ("validation", "test_unseen"),\n    ):\n        for field in ("problem_id", "group_id", "normalized_hash"):\n            assert not (\n                {r[field] for r in splits[left]} & {r[field] for r in splits[right]}\n            )\n    return splits, {\n        "conflicting_provenance_rows_removed": conflicts,\n        "same_label_duplicate_rows_removed": len(records) - len(unique) - conflicts,\n        "heldout_rows_excluded_from_train_validation": excluded_heldout,\n        "independent_groups": len(groups),\n    }\n\n\ndef prepare(output, seed=42, heldout="LLAMA"):\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    raw = output / "raw"\n    raw.mkdir(exist_ok=True)\n    checksums = {}\n    for name in (*FILES, "README.md"):\n        path = raw / name\n        if not path.exists():\n            relative = name if name == "README.md" else f"data/{name}"\n            url = (\n                f"https://huggingface.co/datasets/{REPO}/resolve/{REVISION}/{relative}"\n            )\n            with urllib.request.urlopen(url, timeout=120) as response:\n                content = response.read(30_000_001)\n            if len(content) > 30_000_000:\n                raise ValueError("Unexpected download size")\n            path.write_bytes(content)\n        checksums[name] = hashlib.sha256(path.read_bytes()).hexdigest()\n        if checksums[name] != EXPECTED_SHA256[name]:\n            raise ValueError(\n                f"Checksum mismatch: {path}. Do not use a changed dataset silently."\n            )\n    records, excluded = read_sources(raw)\n    splits, audit = make_splits(records, seed, heldout)\n    for name, rows in splits.items():\n        (output / f"{name}.jsonl").write_text(\n            "".join(json.dumps(row, ensure_ascii=False) + "\\n" for row in rows),\n            encoding="utf-8",\n        )\n    report = {\n        "dataset": REPO,\n        "revision": REVISION,\n        "license_declared_by_publisher": "CDLA-Permissive-2.0",\n        "seed": seed,\n        "heldout_generator": heldout,\n        "raw_sha256": checksums,\n        "excluded": excluded,\n        "audit": audit,\n        "splits": {\n            name: {\n                "rows": len(rows),\n                "labels": dict(Counter(r["label"] for r in rows)),\n                "generators": dict(Counter(r["generator"] for r in rows)),\n                "problems": len({r["problem_id"] for r in rows}),\n            }\n            for name, rows in splits.items()\n        },\n        "limitations": [\n            "test_seen and test_unseen reuse human test controls; do not pool them",\n            "No author-disjoint guarantee or pretrained-model contamination check",\n            "Token-normalized dedup is not full semantic near-duplicate detection",\n            "Dataset provenance labels are publisher assertions, not proof of authorship",\n            "AI repair/hybrid code excluded; no claim on mixed human/AI code",\n        ],\n    }\n    (output / "data_report.json").write_text(\n        json.dumps(report, indent=2), encoding="utf-8"\n    )\n    return report\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--output", default="experiments/codebert/data/aigcodeset")\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--heldout", default="LLAMA", choices=sorted(GENERATORS))\n    args = parser.parse_args()\n    print(json.dumps(prepare(args.output, args.seed, args.heldout), indent=2))\n', encoding="utf-8")


In [ ]:
(ROOT / 'evaluation.py').write_text('"""Dependency-free threshold and abstention reports; not authorship probabilities."""\n\nimport math\n\n\ndef wilson_upper(errors, total, z=1.96):\n    if not total:\n        return None\n    p = errors / total\n    return (\n        p\n        + z * z / (2 * total)\n        + z * math.sqrt(p * (1 - p) / total + z * z / (4 * total * total))\n    ) / (1 + z * z / total)\n\n\ndef binary_metrics(labels, scores, threshold=0.5):\n    if len(labels) != len(scores) or not labels or set(labels) != {0, 1}:\n        raise ValueError("Expected equal nonempty arrays containing both labels")\n    if any(not math.isfinite(float(p)) or not 0 <= p <= 1 for p in scores):\n        raise ValueError("Scores must be finite and in [0,1]")\n    tn = fp = fn = tp = 0\n    for y, p in zip(labels, scores):\n        if y == 0:\n            fp += p >= threshold\n            tn += p < threshold\n        else:\n            tp += p >= threshold\n            fn += p < threshold\n    precision = tp / (tp + fp) if tp + fp else 0.0\n    recall = tp / (tp + fn)\n    return {\n        "threshold": threshold,\n        "n": len(labels),\n        "confusion_matrix": [[tn, fp], [fn, tp]],\n        "accuracy": (tn + tp) / len(labels),\n        "precision_ai": precision,\n        "recall_ai": recall,\n        "f1_ai": 2 * precision * recall / (precision + recall)\n        if precision + recall\n        else 0.0,\n        "human_false_positive_rate": fp / (tn + fp),\n        "human_fpr_wilson95_upper": wilson_upper(fp, tn + fp),\n    }\n\n\ndef select_threshold(labels, scores, max_fpr=0.01):\n    # Include an explicit \'never flag\' option; ties favor a higher threshold.\n    candidates = sorted({0.0, 1.000001, *map(float, scores)})\n    reports = [binary_metrics(labels, scores, t) for t in candidates]\n    feasible = [m for m in reports if m["human_false_positive_rate"] <= max_fpr]\n    return max(\n        feasible,\n        key=lambda m: (m["recall_ai"], -m["human_false_positive_rate"], m["threshold"]),\n    )\n\n\ndef abstention_metrics(labels, scores, low=0.35, high=0.65):\n    binary_metrics(labels, scores)  # Validate inputs.\n    human_n = labels.count(0)\n    ai_n = labels.count(1)\n    inconclusive = sum(low < p < high for p in scores)\n    return {\n        "low": low,\n        "high": high,\n        "inconclusive_rate": inconclusive / len(labels),\n        "human_flagged_ai_rate": sum(\n            y == 0 and p >= high for y, p in zip(labels, scores)\n        )\n        / human_n,\n        "ai_called_human_rate": sum(y == 1 and p <= low for y, p in zip(labels, scores))\n        / ai_n,\n    }\n', encoding="utf-8")


In [ ]:
(ROOT / 'train.py').write_text('"""Single-source authorship experiment. Never executes dataset code or modifies backend."""\n\nimport argparse\nimport hashlib\nimport importlib.metadata\nimport json\nimport platform\nimport random\nimport shutil\nimport time\nfrom pathlib import Path\n\nfrom evaluation import abstention_metrics, binary_metrics, select_threshold\n\nMODEL = "microsoft/codebert-base"\nMODEL_REVISION = "3b0952feddeffad0063f274080e3c23d75e7eb39"\n\n\ndef read_jsonl(path):\n    return [\n        json.loads(line)\n        for line in Path(path).read_text(encoding="utf-8").splitlines()\n        if line.strip()\n    ]\n\n\ndef train(data_dir, output, epochs=3, batch_size=4, seed=42, allow_cpu=False):\n    import numpy as np\n    import torch\n    from sklearn.feature_extraction.text import TfidfVectorizer\n    from sklearn.linear_model import LogisticRegression\n    from sklearn.metrics import average_precision_score, roc_auc_score\n    from torch.utils.data import DataLoader\n    from transformers import AutoModelForSequenceClassification, AutoTokenizer\n\n    output = Path(output)\n    if output.exists() and any(output.iterdir()):\n        raise ValueError(\n            "Output must be empty; use a new run directory to preserve earlier results"\n        )\n    if not torch.cuda.is_available() and not allow_cpu:\n        raise RuntimeError(\n            "Enable Kaggle GPU. CPU training requires explicit --allow-cpu"\n        )\n    if epochs < 1 or batch_size < 1:\n        raise ValueError("epochs and batch_size must be positive")\n    output.mkdir(parents=True, exist_ok=True)\n    started = time.time()\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    torch.backends.cudnn.deterministic = True\n    torch.backends.cudnn.benchmark = False\n    data = {\n        s: read_jsonl(Path(data_dir) / f"{s}.jsonl")\n        for s in ("train", "validation", "test_seen", "test_unseen")\n    }\n    manifest = json.loads(\n        (Path(data_dir) / "data_report.json").read_text(encoding="utf-8")\n    )\n    for left, right in (\n        ("train", "validation"),\n        ("train", "test_seen"),\n        ("train", "test_unseen"),\n        ("validation", "test_seen"),\n        ("validation", "test_unseen"),\n    ):\n        for field in ("group_id", "problem_id", "normalized_hash"):\n            if {r[field] for r in data[left]} & {r[field] for r in data[right]}:\n                raise ValueError(f"Leakage: {left}/{right}/{field}")\n    if any(\n        r["generator"] == manifest["heldout_generator"]\n        for s in ("train", "validation")\n        for r in data[s]\n    ):\n        raise ValueError("Held-out generator found in training or validation")\n\n    tokenizer = AutoTokenizer.from_pretrained(\n        MODEL, revision=MODEL_REVISION, trust_remote_code=False\n    )\n    max_length = 512  # Same single-source truncation as the current backend.\n\n    def collate(rows):\n        encoded = tokenizer(\n            [r["code"] for r in rows],\n            padding=True,\n            truncation=True,\n            max_length=max_length,\n            return_tensors="pt",\n        )\n        return encoded, torch.tensor([r["label"] for r in rows], dtype=torch.long)\n\n    loaders = {\n        s: DataLoader(\n            rows,\n            batch_size=batch_size,\n            shuffle=s == "train",\n            num_workers=0,\n            collate_fn=collate,\n        )\n        for s, rows in data.items()\n    }\n    model = AutoModelForSequenceClassification.from_pretrained(\n        MODEL,\n        revision=MODEL_REVISION,\n        num_labels=2,\n        trust_remote_code=False,\n        id2label={0: "human", 1: "ai_generated"},\n        label2id={"human": 0, "ai_generated": 1},\n    ).to(device)\n    model.config.ai_detector_task = "single_source_origin_classification"\n    model.config.ai_detector_max_length = max_length\n    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)\n    counts = np.bincount([r["label"] for r in data["train"]], minlength=2)\n    if min(counts) == 0:\n        raise ValueError("Training data lacks a class")\n    weights = torch.tensor(\n        len(data["train"]) / (2 * counts), dtype=torch.float32, device=device\n    )\n    loss_fn = torch.nn.CrossEntropyLoss(weight=weights)\n    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")\n\n    @torch.no_grad()\n    def predict(loader):\n        model.eval()\n        scores = []\n        for features, _ in loader:\n            logits = model(**{k: v.to(device) for k, v in features.items()}).logits\n            scores.extend(torch.softmax(logits, dim=-1)[:, 1].cpu().tolist())\n        return scores\n\n    history, best_ap = [], -1.0\n    best_dir = output / "best_model"\n    for epoch in range(epochs):\n        model.train()\n        losses = []\n        for index, (features, labels) in enumerate(loaders["train"]):\n            optimizer.zero_grad(set_to_none=True)\n            with torch.autocast(\n                device_type=device.type,\n                dtype=torch.float16,\n                enabled=device.type == "cuda",\n            ):\n                logits = model(**{k: v.to(device) for k, v in features.items()}).logits\n                loss = loss_fn(logits, labels.to(device))\n            scaler.scale(loss).backward()\n            scaler.unscale_(optimizer)\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n            scaler.step(optimizer)\n            scaler.update()\n            losses.append(loss.item())\n            if index % 100 == 0:\n                print(\n                    f"epoch={epoch + 1} batch={index}/{len(loaders[\'train\'])} loss={loss.item():.4f}",\n                    flush=True,\n                )\n        val_prob = predict(loaders["validation"])\n        labels = [r["label"] for r in data["validation"]]\n        ap = float(average_precision_score(labels, val_prob))\n        row = {\n            "epoch": epoch + 1,\n            "train_loss": float(np.mean(losses)),\n            "validation_average_precision": ap,\n            "validation_at_05": binary_metrics(labels, val_prob),\n        }\n        history.append(row)\n        print(json.dumps(row), flush=True)\n        if ap > best_ap:\n            best_ap = ap\n            model.save_pretrained(best_dir, safe_serialization=True)\n            tokenizer.save_pretrained(best_dir)\n        (output / "history.json").write_text(\n            json.dumps(history, indent=2), encoding="utf-8"\n        )\n\n    del optimizer, scaler, model\n    if device.type == "cuda":\n        torch.cuda.empty_cache()\n    model = AutoModelForSequenceClassification.from_pretrained(\n        best_dir, local_files_only=True\n    ).to(device)\n    val_prob = predict(loaders["validation"])\n    val_labels = [r["label"] for r in data["validation"]]\n    chosen = select_threshold(val_labels, val_prob, max_fpr=0.01)\n\n    # Baseline trained on identical rows, never fit on validation/test.\n    vectorizer = TfidfVectorizer(\n        analyzer="char", ngram_range=(3, 5), max_features=30000, min_df=2\n    )\n    baseline = LogisticRegression(\n        class_weight="balanced", max_iter=1000, random_state=seed\n    )\n    baseline.fit(\n        vectorizer.fit_transform([r["code"] for r in data["train"]]),\n        [r["label"] for r in data["train"]],\n    )\n    baseline_val = baseline.predict_proba(\n        vectorizer.transform([r["code"] for r in data["validation"]])\n    )[:, 1].tolist()\n    baseline_threshold = select_threshold(val_labels, baseline_val)["threshold"]\n\n    def report(rows, scores, threshold):\n        labels = [r["label"] for r in rows]\n        return {\n            "binary_at_05": binary_metrics(labels, scores),\n            "validation_selected_threshold": binary_metrics(labels, scores, threshold),\n            "current_backend_035_065": abstention_metrics(labels, scores),\n            "average_precision": float(average_precision_score(labels, scores)),\n            "roc_auc": float(roc_auc_score(labels, scores)),\n        }\n\n    results = {\n        "validation_selection": chosen,\n        "codebert": {},\n        "tfidf_logistic": {},\n        "per_generator": {},\n    }\n    for split in ("validation", "test_seen", "test_unseen"):\n        rows = data[split]\n        prob = val_prob if split == "validation" else predict(loaders[split])\n        results["codebert"][split] = report(rows, prob, chosen["threshold"])\n        base_prob = baseline.predict_proba(\n            vectorizer.transform([r["code"] for r in rows])\n        )[:, 1].tolist()\n        results["tfidf_logistic"][split] = report(rows, base_prob, baseline_threshold)\n        generators = sorted({r["generator"] for r in rows if r["label"] == 1})\n        results["per_generator"][split] = {}\n        for generator in generators:\n            indices = [\n                i\n                for i, r in enumerate(rows)\n                if r["label"] == 0 or r["generator"] == generator\n            ]\n            results["per_generator"][split][generator] = report(\n                [rows[i] for i in indices],\n                [prob[i] for i in indices],\n                chosen["threshold"],\n            )\n        predictions = [\n            {\n                "record_id": r["record_id"],\n                "problem_id": r["problem_id"],\n                "label": r["label"],\n                "generator": r["generator"],\n                "ai_score": p,\n                "prediction": int(p >= chosen["threshold"]),\n                "truncated": len(tokenizer.encode(r["code"], truncation=False))\n                > max_length,\n            }\n            for r, p in zip(rows, prob)\n        ]\n        (output / f"{split}_predictions.jsonl").write_text(\n            "".join(json.dumps(r) + "\\n" for r in predictions), encoding="utf-8"\n        )\n        results["codebert"][split]["truncated_rows"] = sum(\n            r["truncated"] for r in predictions\n        )\n    metadata = {\n        "model": MODEL,\n        "model_revision": MODEL_REVISION,\n        "seed": seed,\n        "epochs": epochs,\n        "batch_size": batch_size,\n        "max_length": max_length,\n        "learning_rate": 2e-5,\n        "python": platform.python_version(),\n        "device": str(device),\n        "gpu": torch.cuda.get_device_name() if device.type == "cuda" else None,\n        "packages": {\n            p: importlib.metadata.version(p)\n            for p in ("torch", "transformers", "scikit-learn", "numpy")\n        },\n        "data_report": manifest,\n        "seconds": time.time() - started,\n        "split_sha256": {\n            s: hashlib.sha256((Path(data_dir) / f"{s}.jsonl").read_bytes()).hexdigest()\n            for s in data\n        },\n        "limitations": [\n            "Scores are not calibrated probabilities of authorship",\n            "1% empirical validation FPR is not a deployment guarantee; inspect confidence bound",\n            "Backend still uses 0.35/0.65; selected binary threshold is evaluation only",\n            "No deployment approval; no penalty or academic misconduct decision from this signal",\n        ],\n    }\n    (output / "metrics.json").write_text(\n        json.dumps(results, indent=2), encoding="utf-8"\n    )\n    (output / "run_manifest.json").write_text(\n        json.dumps(metadata, indent=2), encoding="utf-8"\n    )\n    # A self-contained offline-loadable model. Metrics intentionally outside model directory.\n    model.cpu()\n    model.save_pretrained(best_dir, safe_serialization=True)\n    shutil.copy2(Path(data_dir) / "raw" / "README.md", output / "DATASET_CARD.md")\n    archive = Path(\n        shutil.make_archive(str(output / "ai_detector_model"), "zip", root_dir=best_dir)\n    )\n    (output / "model_sha256.txt").write_text(\n        hashlib.sha256(archive.read_bytes()).hexdigest() + "\\n", encoding="utf-8"\n    )\n    print(\n        f"Finished. Inspect {output / \'metrics.json\'} BEFORE integrating the checkpoint."\n    )\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--data", required=True)\n    parser.add_argument("--output", required=True)\n    parser.add_argument("--epochs", type=int, default=3)\n    parser.add_argument("--batch-size", type=int, default=4)\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--allow-cpu", action="store_true")\n    args = parser.parse_args()\n    train(\n        args.data, args.output, args.epochs, args.batch_size, args.seed, args.allow_cpu\n    )\n', encoding="utf-8")


In [ ]:
(ROOT / 'test_pipeline.py').write_text('import tempfile\nimport unittest\nfrom pathlib import Path\n\nfrom evaluation import (\n    abstention_metrics,\n    binary_metrics,\n    select_threshold,\n    wilson_upper,\n)\nfrom prepare_data import clean_code, digest, make_splits, normalized_hash, read_sources\n\n\ndef synthetic_records():\n    rows = []\n    for p in range(100):\n        for generator, label in (\n            ("HUMAN", 0),\n            ("GEMINI", 1),\n            ("CODESTRAL", 1),\n            ("LLAMA", 1),\n        ):\n            code = f"print({p}, {generator!r})\\n"\n            rows.append(\n                {\n                    "record_id": f"{p}:{generator}",\n                    "code": code,\n                    "label": label,\n                    "problem_id": f"p{p}",\n                    "generator": generator,\n                    "sha256": digest(code),\n                    "normalized_hash": normalized_hash(code),\n                }\n            )\n    return rows\n\n\nclass PipelineTests(unittest.TestCase):\n    def test_normalize_and_keep_syntax_errors(self):\n        self.assertEqual(clean_code("```python\\nprint(1)\\n```"), "print(1)\\n")\n        self.assertEqual(\n            normalized_hash("print(1) # comment\\n"), normalized_hash("print ( 1 )\\n")\n        )\n        self.assertEqual(len(normalized_hash("def broken(:")), 64)\n\n    def test_splits_are_reproducible_and_group_disjoint(self):\n        splits, _ = make_splits(synthetic_records())\n        self.assertEqual(splits, make_splits(synthetic_records())[0])\n        for left, right in (\n            ("train", "validation"),\n            ("train", "test_seen"),\n            ("validation", "test_unseen"),\n        ):\n            self.assertFalse(\n                {r["problem_id"] for r in splits[left]}\n                & {r["problem_id"] for r in splits[right]}\n            )\n        self.assertNotIn(\n            "LLAMA", {r["generator"] for r in splits["train"] + splits["validation"]}\n        )\n        self.assertEqual(\n            {r["generator"] for r in splits["test_unseen"]}, {"HUMAN", "LLAMA"}\n        )\n\n    def test_conflicting_labels_removed(self):\n        rows = synthetic_records()\n        conflict = dict(rows[0], record_id="conflict", label=1, generator="GEMINI")\n        splits, audit = make_splits(rows + [conflict])\n        self.assertEqual(audit["conflicting_provenance_rows_removed"], 2)\n        self.assertNotIn(\n            rows[0]["normalized_hash"],\n            {r["normalized_hash"] for part in splits.values() for r in part},\n        )\n\n    def test_duplicates_link_problem_groups(self):\n        rows = synthetic_records()\n        rows.append(dict(rows[0], problem_id="p99", record_id="linked"))\n        splits, audit = make_splits(rows)\n        self.assertEqual(audit["independent_groups"], 99)\n        partitions = {}\n        for s, rs in splits.items():\n            for r in rs:\n                partitions[r["problem_id"]] = "test" if s.startswith("test") else s\n        self.assertEqual(partitions["p0"], partitions["p99"])\n\n    def test_hybrid_excluded_and_labels_validated(self):\n        import csv\n\n        with tempfile.TemporaryDirectory() as tmp:\n            root = Path(tmp)\n            fields = [\n                "problem_id",\n                "submission_id",\n                "LLM",\n                "status_in_folder",\n                "code",\n                "label",\n            ]\n            for file, rows in [\n                (\n                    "human_selected_dataset.csv",\n                    [["p1", "s1", "Human", "Accepted", "print(1)", 0]],\n                ),\n                (\n                    "created_dataset_with_llms.csv",\n                    [\n                        ["p1", "s1", "GEMINI", "Runtime", "print(2)", 1],\n                        ["p1", "unrelated", "GEMINI", "Generate", "print(3)", 1],\n                    ],\n                ),\n            ]:\n                with (root / file).open("w", newline="", encoding="utf-8") as f:\n                    writer = csv.writer(f)\n                    writer.writerow(fields)\n                    writer.writerows(rows)\n            records, excluded = read_sources(root)\n            self.assertEqual(len(records), 2)\n            self.assertEqual(excluded["ai_repair_or_hybrid"], 1)\n\n    def test_fpr_and_abstention(self):\n        result = binary_metrics([0, 0, 1, 1], [0.1, 0.8, 0.4, 0.9])\n        self.assertEqual(result["confusion_matrix"], [[1, 1], [1, 1]])\n        self.assertEqual(result["human_false_positive_rate"], 0.5)\n        self.assertGreater(wilson_upper(0, 20), 0)\n        self.assertEqual(\n            abstention_metrics([0, 0, 1, 1], [0.1, 0.8, 0.4, 0.9])["inconclusive_rate"],\n            0.25,\n        )\n\n    def test_threshold_can_abstain_from_all_flags(self):\n        selected = select_threshold([0, 1], [1.0, 0.5])\n        self.assertGreater(selected["threshold"], 1.0)\n        self.assertEqual(selected["recall_ai"], 0.0)\n\n    def test_threshold_rejects_invalid_inputs(self):\n        for labels, scores in (\n            ([0, 1], [float("nan"), 0.2]),\n            ([0], [0.3]),\n            ([0, 1], [0.2]),\n        ):\n            with self.assertRaises(ValueError):\n                select_threshold(labels, scores)\n\n\nif __name__ == "__main__":\n    unittest.main()\n', encoding="utf-8")


## 1. Kiểm thử pipeline, tải dữ liệu công khai và kiểm tra chia tập
Dữ liệu chỉ được đọc dưới dạng văn bản; không chạy code trong dataset.

In [ ]:
import subprocess

subprocess.run([sys.executable, "-m", "unittest", "test_pipeline", "-v"], cwd=ROOT, check=True)
from prepare_data import prepare

DATA = ROOT / "data"
report = prepare(DATA, seed=42, heldout="LLAMA")
print(json.dumps(report, indent=2))


## 2. Fine-tune
Batch 4, 3 epoch. Nếu GPU thiếu RAM: giảm BATCH_SIZE xuống 2. Không chỉnh tham số theo kết quả test; dùng validation. Mỗi lần chạy tạo thư mục riêng, không ghi đè checkpoint cũ.

In [ ]:
import torch

assert torch.cuda.is_available(), "Hãy bật GPU trong Settings của Kaggle"
print(torch.cuda.get_device_name(0))
EPOCHS, BATCH_SIZE, SEED = 3, 4, 42
RUN = ROOT / ("run_" + time.strftime("%Y%m%d_%H%M%S"))
subprocess.run([sys.executable, str(ROOT/"train.py"), "--data", str(DATA), "--output", str(RUN), "--epochs", str(EPOCHS), "--batch-size", str(BATCH_SIZE), "--seed", str(SEED)], check=True)


## 3. Kết quả và tải về
Xem human_false_positive_rate cùng human_fpr_wilson95_upper, recall_ai, baseline và test_unseen. Ngưỡng chọn trên validation là nhị phân; backend hiện vẫn dùng 0.35/0.65 nên có báo cáo riêng. Không bật model vào demo trước khi review.

In [ ]:
metrics = json.loads((RUN/"metrics.json").read_text())
print(json.dumps(metrics, indent=2))
import shutil

from IPython.display import FileLink, display

RESULTS = ROOT / (RUN.name + "_reports")
RESULTS.mkdir(exist_ok=True)
for path in RUN.iterdir():
    if path.is_file() and path.suffix != ".zip":
        shutil.copy2(path, RESULTS/path.name)
shutil.copy2(DATA/"data_report.json", RESULTS/"data_report.json")
archive = shutil.make_archive(str(RESULTS), "zip", root_dir=RESULTS)
display(FileLink(str(RUN/"ai_detector_model.zip")))
display(FileLink(archive))
print("Nếu FileLink không tải được, dùng tab Output của Kaggle.")
